# NBO resonance smoke tests

Compact checks for allyl charge states, benzene Lewis structures, and acene scaling. The allyl systems are optimized before NBO. The acene timing series uses direct SMILES geometries and reruns benzene as the first scaling point.

In [1]:
import time

import veloxchem as vlx


def make_molecule(smiles, charge=0, multiplicity=1):
    molecule = vlx.Molecule.read_smiles(smiles)
    molecule.set_charge(charge)
    molecule.set_multiplicity(multiplicity)
    return molecule


def copy_geometry_with_state(reference, charge=0, multiplicity=1):
    molecule = vlx.Molecule.read_xyz_string(reference.get_xyz_string())
    molecule.set_charge(charge)
    molecule.set_multiplicity(multiplicity)
    return molecule


def make_scf_driver(molecule):
    if int(round(float(molecule.get_multiplicity()))) == 1:
        return vlx.ScfRestrictedDriver()
    return vlx.ScfUnrestrictedDriver()


def optimize_molecule(molecule, basis_label='def2-svp', label=None):
    label = 'molecule' if label is None else label
    print(f'\n=== {label}: geometry optimization ===')
    scf_drv = make_scf_driver(molecule)
    basis = vlx.MolecularBasis.read(molecule, basis_label)
    scf = scf_drv.compute(molecule, basis)
    opt_drv = vlx.OptimizationDriver(scf_drv)
    opt_results = opt_drv.compute(molecule, basis, scf)
    optimized = vlx.Molecule.read_xyz_string(opt_results['final_geometry'])
    optimized.set_charge(molecule.get_charge())
    optimized.set_multiplicity(molecule.get_multiplicity())
    return optimized


def run_scf_nbo(molecule, basis_label='def2-svp', max_alternatives=None, constraints=None, label=None):
    label = 'molecule' if label is None else label
    print(f'\n=== {label}: SCF ===')
    scf_drv = make_scf_driver(molecule)
    basis = vlx.MolecularBasis.read(molecule, basis_label)
    scf_drv.compute(molecule, basis)

    print(f'\n=== {label}: NBO ===')
    nbo_drv = vlx.NboDriver()
    options = {}
    if max_alternatives is not None:
        options['max_alternatives'] = int(max_alternatives)

    start = time.perf_counter()
    nbo_results = nbo_drv.compute(
        molecule,
        basis,
        scf_drv.mol_orbs,
        options=options,
        constraints=constraints,
    )
    elapsed = time.perf_counter() - start
    return nbo_drv, nbo_results, elapsed


def resonance_weight_rows(label, results):
    rows = []
    for alternative in results.get('alternatives', []):
        pi_bonds = ', '.join(
            f"{pair[0]}-{pair[1]}" for pair in alternative.get('pi_bonds', [])
        ) or 'none'
        one_e = ', '.join(str(atom) for atom in alternative.get('active_one_electron_atoms', [])) or 'none'
        lp = ', '.join(str(atom) for atom in alternative.get('active_lone_pair_atoms', [])) or 'none'
        pos = ', '.join(str(atom) for atom in alternative.get('active_positive_atoms', [])) or 'none'
        rows.append({
            'system': label,
            'rank': int(alternative.get('rank', 0)),
            'weight': f"{float(alternative.get('weight', 0.0)):.5f}",
            'pi_bonds': pi_bonds,
            'one_e': one_e,
            'lp': lp,
            'pos': pos,
        })
    return rows


def print_table(rows, columns):
    if not rows:
        print('(no rows)')
        return
    widths = {
        column: max(len(column), *(len(f"{row.get(column, '')}") for row in rows))
        for column in columns
    }
    print('  '.join(column.ljust(widths[column]) for column in columns))
    print('  '.join('-' * widths[column] for column in columns))
    for row in rows:
        print('  '.join(f"{row.get(column, '')}".ljust(widths[column]) for column in columns))

## Allyl cation, radical, and anion

The allyl cation is optimized first and its geometry is reused as the starting point for the radical and anion optimizations. This keeps the three charge/spin states on a common atom ordering and avoids comparing unrelated SMILES embeddings. Each state is then run with the same three requested pi patterns: `1-2`, `2-3`, and `1-3`.

In [2]:
allyl_constraints = vlx.NboConstraints(
    allowed_pi_bonds=((1, 2), (2, 3), (1, 3)),
)
allyl_cases = [
    ('allyl cation', 1, 1),
    ('allyl radical', 0, 2),
    ('allyl anion', -1, 1),
]

allyl_cation_start = make_molecule('[CH2+]C=C', charge=1, multiplicity=1)
allyl_cation_opt = optimize_molecule(allyl_cation_start, label='allyl cation')

allyl_results = {}
allyl_rows = []
for label, charge, multiplicity in allyl_cases:
    if label == 'allyl cation':
        start_geometry = allyl_cation_opt
    else:
        start_geometry = copy_geometry_with_state(
            allyl_cation_opt,
            charge=charge,
            multiplicity=multiplicity,
        )
        start_geometry = optimize_molecule(start_geometry, label=label)

    nbo_drv, results, elapsed = run_scf_nbo(
        start_geometry,
        max_alternatives=3,
        constraints=allyl_constraints,
        label=label,
    )
    allyl_results[label] = (nbo_drv, results)
    allyl_rows.extend(resonance_weight_rows(label, results))
    print(f'{label}: NBO wall time {elapsed:.3f} s')

print('\nallyl three-structure weights')
print_table(allyl_rows, ['system', 'rank', 'weight', 'pi_bonds', 'one_e', 'lp', 'pos'])

for label, (nbo_drv, results) in allyl_results.items():
    print(f'\n{label}')
    nbo_drv.show_structures(results)


=== allyl cation: geometry optimization ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                       

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


allyl radical


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


allyl anion


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Benzene Kekule and Dewar structures

Benzene is kept separate from the acene scaling loop. This cell runs unconstrained NBO, the two Kekule patterns, and Kekule plus Dewar-type transannular pi pairs.

In [3]:
benzene = make_molecule('c1ccccc1')
benzene_drv, benzene_nbo, benzene_time = run_scf_nbo(
    benzene,
    label='benzene unconstrained',
)
print(f'benzene unconstrained NBO wall time: {benzene_time:.3f} s')
benzene_drv.show_structures(benzene_nbo)

benzene_kekule_constraints = vlx.NboConstraints(
    allowed_pi_bonds=((1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 1)),
)
benzene_kekule_drv, benzene_kekule_nbo, _ = run_scf_nbo(
    benzene,
    max_alternatives=2,
    constraints=benzene_kekule_constraints,
    label='benzene Kekule',
)
print('\nbenzene Kekule structures')
print_table(
    resonance_weight_rows('benzene Kekule', benzene_kekule_nbo),
    ['system', 'rank', 'weight', 'pi_bonds'],
)
benzene_kekule_drv.show_structures(benzene_kekule_nbo)

benzene_kekule_dewar_constraints = vlx.NboConstraints(
    allowed_pi_bonds=(
        (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 1),
        (1, 4), (2, 5), (3, 6),
    ),
)
benzene_dewar_drv, benzene_dewar_nbo, _ = run_scf_nbo(
    benzene,
    max_alternatives=5,
    constraints=benzene_kekule_dewar_constraints,
    label='benzene Kekule + Dewar',
)
print('\nbenzene Kekule + Dewar structures')
print_table(
    resonance_weight_rows('benzene Dewar', benzene_dewar_nbo),
    ['system', 'rank', 'weight', 'pi_bonds'],
)
benzene_dewar_drv.show_structures(benzene_dewar_nbo)


=== benzene unconstrained: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


=== benzene Kekule: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                       

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


=== benzene Kekule + Dewar: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                               

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Acene scaling without geometry optimization

This loop reruns benzene and continues through naphthalene, anthracene, tetracene, and pentacene. It uses direct SMILES geometries, prints progress for each molecule, leaves SCF/NBO output visible, and reports unconstrained NBO wall times.

In [4]:
acene_smiles = {
    'benzene': 'c1ccccc1',
    'naphthalene': 'c1ccc2ccccc2c1',
    'anthracene': 'c1ccc2cc3ccccc3cc2c1',
    'tetracene': 'c1ccc2cc3cc4ccccc4cc3cc2c1',
    'pentacene': 'c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1',
}

acene_rows = []
acene_results = {}
for label, smiles in acene_smiles.items():
    print(f'\n### Running acene scaling point: {label} ###')
    molecule = make_molecule(smiles)
    nbo_drv, results, elapsed = run_scf_nbo(
        molecule,
        label=f'{label} unconstrained',
    )
    acene_results[label] = (nbo_drv, results)
    acene_rows.append({
        'system': label,
        'atoms': molecule.number_of_atoms(),
        'nbo_time_s': f'{elapsed:.3f}',
        'alternatives': len(results.get('alternatives', [])),
    })

print('\nunconstrained NBO timing for acenes')
print_table(acene_rows, ['system', 'atoms', 'nbo_time_s', 'alternatives'])


### Running acene scaling point: benzene ###

=== benzene unconstrained: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10  

## Acene Lewis structures

Show the unconstrained Lewis structures from the acene scaling run.

In [5]:
for label, (nbo_drv, results) in acene_results.items():
    print(f'\n{label}')
    nbo_drv.show_structures(results)


benzene


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


naphthalene


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


anthracene


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


tetracene


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


pentacene


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
coronene_smiles = 'C1=CC2=C3C4=C1C=CC5=C4C6=C(C=C5)C=CC7=C6C3=C(C=C2)C=C7'
coronene = make_molecule(coronene_smiles)

coronene_drv, coronene_nbo, coronene_time = run_scf_nbo(
    coronene,
    label='coronene unconstrained',
)

print('\ncoronene unconstrained NBO timing')
print_table(
    [{
        'system': 'coronene',
        'atoms': coronene.number_of_atoms(),
        'nbo_time_s': f'{coronene_time:.3f}',
        'alternatives': len(coronene_nbo.get('alternatives', [])),
    }],
    ['system', 'atoms', 'nbo_time_s', 'alternatives'],
)
coronene_drv.show_structures(coronene_nbo)


=== coronene unconstrained: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                               

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
trinitrobenzene_smiles = 'O=[N+]([O-])c1cc([N+](=O)[O-])cc([N+](=O)[O-])c1'
trinitrobenzene = make_molecule(trinitrobenzene_smiles)

expected_trinitrobenzene_structures = 2 * 2**3
print(f'Expected simple formal resonance structures: {expected_trinitrobenzene_structures}')

trinitrobenzene_drv, trinitrobenzene_nbo, trinitrobenzene_time = run_scf_nbo(
    trinitrobenzene,
    max_alternatives=expected_trinitrobenzene_structures,
    label='1,3,5-trinitrobenzene unconstrained',
)

print('\n1,3,5-trinitrobenzene unconstrained NBO timing')
print_table(
    [{
        'system': '1,3,5-trinitrobenzene',
        'atoms': trinitrobenzene.number_of_atoms(),
        'nbo_time_s': f'{trinitrobenzene_time:.3f}',
        'alternatives': len(trinitrobenzene_nbo.get('alternatives', [])),
    }],
    ['system', 'atoms', 'nbo_time_s', 'alternatives'],
)

print('\n1,3,5-trinitrobenzene resonance structures')
print_table(
    resonance_weight_rows('1,3,5-trinitrobenzene', trinitrobenzene_nbo),
    ['system', 'rank', 'weight', 'pi_bonds', 'lp', 'pos'],
)
trinitrobenzene_drv.show_structures(trinitrobenzene_nbo)

Expected simple formal resonance structures: 16

=== 1,3,5-trinitrobenzene unconstrained: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error V

* Info * Starting Reduced Basis SCF calculation...                                                                        
* Info * ...done. SCF energy in reduced basis set: -839.906884978005 a.u. Time: 0.95 sec.                                 
                                                                                                                          
                                                                                                                          
               Iter. | Hartree-Fock Energy | Energy Change | Gradient Norm | Max. Gradient | Density Change               
               --------------------------------------------------------------------------------------------               
                  1      -840.261970467096    0.0000000000      0.32563392      0.01396600      0.00000000                
                  2      -840.271864660168   -0.0098941931      0.20881783      0.00965015      0.20292478                
                

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
porphyrin_xyz = """38

H 3.209599995 3.172309777 0.000000000
N 0.000000000 -2.113600613 0.000000000
N 2.031813914 0.000000000 0.000000000
N 0.000000000 2.113600613 0.000000000
N -2.031813914 0.000000000 0.000000000
C 1.126850920 -2.888627221 0.000000000
C -1.126850920 -2.888627221 0.000000000
C 1.126850920 2.888627221 0.000000000
C -1.126850920 2.888627221 0.000000000
C 2.850160984 -1.083939750 0.000000000
C 2.850160984 1.083939750 0.000000000
C -2.850160984 -1.083939750 0.000000000
C -2.850160984 1.083939750 0.000000000
C 0.683827200 -4.249372023 0.000000000
C -0.683827200 -4.249372023 0.000000000
C 0.683827200 4.249372023 0.000000000
C -0.683827200 4.249372023 0.000000000
C 4.248378966 -0.675836785 0.000000000
C 4.248378966 0.675836785 0.000000000
C -4.248378966 -0.675836785 0.000000000
C -4.248378966 0.675836785 0.000000000
C 2.433989580 -2.416542873 0.000000000
C -2.433989580 -2.416542873 0.000000000
C -2.433989580 2.416542873 0.000000000
C 2.433989580 2.416542873 0.000000000
H 1.341416542 -5.103998738 0.000000000
H -1.341416542 -5.103998738 0.000000000
H 1.341416542 5.103998738 0.000000000
H -1.341416542 5.103998738 0.000000000
H 5.096138011 -1.344322017 0.000000000
H 5.096138011 1.344322017 0.000000000
H -5.096138011 -1.344322017 0.000000000
H -5.096138011 1.344322017 0.000000000
H 3.209599995 -3.172309777 0.000000000
H -3.209599995 -3.172309777 0.000000000
H -3.209599995 3.172309777 0.000000000
H 0.000000000 -1.102285542 0.000000000
H 0.000000000 1.102285542 0.000000000
"""

porphyrin = vlx.Molecule.read_xyz_string(porphyrin_xyz)
porphyrin.set_charge(0)
porphyrin.set_multiplicity(1)   
porphyrin_drv, porphyrin_nbo, porphyrin_time = run_scf_nbo(
    porphyrin,
    label='porphyrin unconstrained',
)
print('\nporphyrin unconstrained NBO timing')
print_table(
    [{
        'system': 'porphyrin',
        'atoms': porphyrin.number_of_atoms(),
        'nbo_time_s': f'{porphyrin_time:.3f}',
        'alternatives': len(porphyrin_nbo.get('alternatives', [])),
    }],
    ['system', 'atoms', 'nbo_time_s', 'alternatives'],
)
print('\nporphyrin resonance structures')
print_table(
    resonance_weight_rows('porphyrin', porphyrin_nbo),
    ['system', 'rank', 'weight', 'pi_bonds', 'lp', 'pos'],
)
porphyrin_drv.show_structures(porphyrin_nbo)


=== porphyrin unconstrained: SCF ===
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                              

3Dmol.js failed to load for some reason. Please check your browser console for error messages.